In [1]:
# =============================================================================
# STEP 7b - FIXING THE MONITOR'S STRUCTURAL DEFECT, AND CORRECTING nb46
#
# Two things prompt this notebook.
#
# CORRECTION. The baseline comparison in nb46 scored each signal on its OWN
# non-NaN subset: the monitor on 19 cells, mean-confidence shift on 18. AUROCs
# computed on different cell sets are not comparable, and the resulting claim that
# a simpler signal beats the monitor was an artefact of that. On the 18 common
# cells both reach 0.900. This notebook supersedes that comparison.
#
# THE REAL DEFECT. The monitor scores classes by WITHIN-ENVIRONMENT percentile
# rank. Some class always ranks highest, so the rule can never say "all clear":
# on CIC-IoT-2023, where no class fails, it flags 5 of 8 at threshold 0.50. That
# is structural, not a tuning problem, and it is the objection a reviewer will
# make. We fix it by scoring against an ABSOLUTE reference rather than against the
# other classes present.
#
# The reference is the permutation null already used for shift measurement: split
# the pooled source-and-target predicted-class scores at random and recompute the
# statistic. A class is flagged when its observed drift exceeds what random
# splitting of its own data produces. That quantity is comparable across
# environments and can return "nothing here".
#
# HONESTY ABOUT POWER. There are 18 usable cells with 5 failing classes. Paired
# bootstraps in nb46 could not distinguish any two signals except raw score KS.
# Any improvement measured here is therefore not distinguishable from noise, and
# the notebook reports ALL variants tried rather than the best one, so the
# selection is visible.
# =============================================================================
import numpy as np, pandas as pd
from scipy import stats
from sklearn.metrics import roc_auc_score
import os, sys, json, shutil, glob, subprocess, hashlib, time
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
assert PROJECT_ROOT.exists(), 'Drive mount unhealthy; restart runtime and remount'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
import importlib
for m in ['config','conformal']:
    if m in sys.modules: importlib.reload(sys.modules[m])
import config
RD=config.REPORTS_DIR; ALPHA=config.ALPHA_PRIMARY
MIN_SUPPORT=20; FAIL_THRESHOLD=0.05; NPERM=200
print('ready | permutation draws per class', NPERM)


Mounted at /content/drive
ready | permutation draws per class 200


In [2]:
# =============================================================================
# Cell 2 - the absolute signal. For each class we compute the observed drift and
# a null distribution obtained by randomly splitting the POOLED source-and-target
# predicted-as-c flows into two halves of the same sizes. The score is the
# standardised excess over that null, which is on a common scale across
# environments and is zero when nothing has moved.
# =============================================================================
def aps_all(P):
    o=np.argsort(-P,axis=1); sp=np.take_along_axis(P,o,1); cum=np.cumsum(sp,1)
    ss=cum-0.5*sp; out=np.empty_like(P); np.put_along_axis(out,o,ss,1); return out

def ks(a,b):
    if len(a)<5 or len(b)<5: return np.nan
    allv=np.sort(np.concatenate([a,b]))
    ca=np.searchsorted(np.sort(a),allv,side='right')/len(a)
    cb=np.searchsorted(np.sort(b),allv,side='right')/len(b)
    return float(np.max(np.abs(ca-cb)))

def absolute_signal(s_src, s_tgt, seed=0, nperm=NPERM):
    """Observed drift minus its permutation null, in null standard deviations.
    Returns (z, observed, null_mean, null_sd). Zero means indistinguishable from
    randomly splitting this class's own data, which is what 'all clear' looks like."""
    if len(s_src) < 5 or len(s_tgt) < 5: return (np.nan,)*4
    obs = ks(s_src, s_tgt)
    pool = np.concatenate([s_src, s_tgt]); n1 = len(s_src)
    rg = np.random.default_rng(seed); null=[]
    for _ in range(nperm):
        perm = rg.permutation(len(pool))
        v = ks(pool[perm[:n1]], pool[perm[n1:]])
        if v == v: null.append(v)
    if len(null) < 20: return (np.nan, obs, np.nan, np.nan)
    mu, sd = float(np.mean(null)), float(np.std(null))
    z = (obs - mu) / sd if sd > 1e-12 else 0.0
    return float(z), float(obs), mu, sd

def entropy(P):
    Q=np.clip(P,1e-12,None); return -(Q*np.log(Q)).sum(1)

def rows_for(dataset, arch, classes, P_src, P_tgt, cov_lookup, seed):
    Ss, St = aps_all(P_src), aps_all(P_tgt)
    yh_s, yh_t = P_src.argmax(1), P_tgt.argmax(1)
    conf_s, conf_t = P_src.max(1), P_tgt.max(1)
    ent_s, ent_t = entropy(P_src), entropy(P_tgt)
    out=[]
    for ci,cn in enumerate(classes):
        ms, mt = yh_s==ci, yh_t==ci
        n_t=int(mt.sum()); low = n_t < MIN_SUPPORT
        share_s, share_t = float(ms.mean()), float(mt.mean())
        z, obs, mu, sd = absolute_signal(Ss[ms,ci], St[mt,ci], seed=seed+ci) if not low else (np.nan,)*4
        # absolute version of the confidence baseline, on the same null footing
        zc = np.nan
        if not low and ms.sum()>=5:
            d_obs = abs(float(conf_s[ms].mean())-float(conf_t[mt].mean()))
            pool=np.concatenate([conf_s[ms], conf_t[mt]]); n1=int(ms.sum())
            rg=np.random.default_rng(seed+1000+ci); nd=[]
            for _ in range(NPERM):
                pm=rg.permutation(len(pool))
                nd.append(abs(pool[pm[:n1]].mean()-pool[pm[n1:]].mean()))
            mu2, sd2 = float(np.mean(nd)), float(np.std(nd))
            zc = (d_obs-mu2)/sd2 if sd2>1e-12 else 0.0
        out.append({'dataset':dataset,'arch':arch,'class':cn,'n_pred_target':n_t,
            'low_support':bool(low),
            'rank_drift':ks(Ss[ms,ci],St[mt,ci]) if not low else np.nan,
            'pred_mass_drop':share_s-share_t,
            'abs_z_drift':z,'obs_drift':obs,'null_mean':mu,'null_sd':sd,
            'abs_z_confidence':zc,
            'b_confidence_shift':abs(float(conf_s[ms].mean())-float(conf_t[mt].mean())) if not low and ms.any() else np.nan,
            'b_entropy_shift':abs(float(ent_s[ms].mean())-float(ent_t[mt].mean())) if not low and ms.any() else np.nan,
            'coverage':cov_lookup.get(cn,np.nan)})
        out[-1]['undercoverage']=(1-ALPHA)-out[-1]['coverage'] if out[-1]['coverage']==out[-1]['coverage'] else np.nan
    return out
print('absolute signal defined: standardised excess over a per-class permutation null')


absolute signal defined: standardised excess over a per-class permutation null


In [3]:
# =============================================================================
# Cell 3 - recompute across all four environments. Fewer models than nb46 because
# the permutation null costs NPERM extra KS evaluations per class; three seeds per
# architecture is ample since the signal is a property of the shift, not the seed.
# =============================================================================
def check_classes(npz, assumed, tag):
    if 'classes' in npz.files:
        saved=[str(x) for x in npz['classes']]
        assert saved==list(assumed), f'{tag}: saved order {saved} != assumed {list(assumed)}'
        return saved
    return list(assumed)

def cov_map(f, rung=None):
    d=pd.read_csv(RD/f); d=d[(np.isclose(d.alpha,ALPHA))&(d.protocol=='SHC')]
    if 'feasible' in d.columns: d=d[d['feasible']]
    if rung is not None and 'rung' in d.columns: d=d[np.isclose(d['rung'],rung)]
    return d.groupby('class')['coverage'].mean().to_dict()

rows=[]; t0=time.time()
CL=config.CANONICAL_CLASSES
te=pd.read_parquet(config.INTERIM_DIR/'nslkdd_test.parquet').reset_index(drop=True)
assign=pd.read_parquet(config.PROC_DIR/'nslkdd_ladder_assignments.parquet')
IDX={(r,j,role):g['test_idx'].to_numpy() for (r,j,role),g in assign.groupby(['rung','realization','role'])}
REALS=sorted(assign[np.isclose(assign.rung,0.80)]['realization'].unique())[:3]
cm=cov_map('coverage_primary_nslkdd.csv',0.80)
for f in sorted(glob.glob(str(config.PROC_DIR/'probs_*.npz')))[:9]:
    arch,seed=Path(f).stem.replace('probs_','').rsplit('_s',1)
    d=np.load(f); P_sp=d['S_pool'].astype(np.float64); P_te=d['target'].astype(np.float64)
    for j in REALS:
        ev=IDX.get((0.80,j,'eval'))
        if ev is None or len(ev)==0: continue
        rows += rows_for('nslkdd',arch,CL,P_sp,P_te[ev],cm,seed=abs(hash((arch,seed,j)))%10000)
print(f'  nslkdd {sum(1 for r in rows if r["dataset"]=="nslkdd")} rows | {time.time()-t0:.0f}s')

UK=['background','dos','scan11','scan44','nerisbotnet']; UCL=sorted(UK)
cm=cov_map('coverage_primary_ugr16.csv')
for f in sorted((config.DATA_DIR/'ugr16_probs').glob('ugr16__*.npz'))[:9]:
    _,arch,sd=Path(f).stem.split('__'); d=np.load(f); cls=check_classes(d,UCL,'ugr16')
    rows += rows_for('ugr16',arch,cls,d['srcpool'].astype(np.float64),d['target'].astype(np.float64),
                     cm,seed=abs(hash((arch,sd)))%10000)
print(f'  ugr16 {sum(1 for r in rows if r["dataset"]=="ugr16")} rows | {time.time()-t0:.0f}s')

CCL=['Benign','DoS']; cm=cov_map('coverage_primary_cicids2017.csv')
for name in ['R1_holdout_Slowhttptest','R3_holdout_GoldenEye','R5_holdout_GoldenEye_Slowloris']:
    for f in sorted((config.DATA_DIR/'cic_probs').glob(f'{name}__*.npz'))[:3]:
        _,arch,sd=Path(f).stem.split('__'); d=np.load(f); cls=check_classes(d,CCL,'cic')
        rows += rows_for('cicids2017',arch,cls,d['srcpool'].astype(np.float64),
                         d['target'].astype(np.float64),cm,seed=abs(hash((name,arch)))%10000)
print(f'  cicids2017 {sum(1 for r in rows if r["dataset"]=="cicids2017")} rows | {time.time()-t0:.0f}s')

ICL=json.loads((RD/'ciciot2023_model_record.json').read_text())['classes_canonical_order']
cm=cov_map('coverage_primary_ciciot2023.csv',0.80)
for f in sorted((config.DATA_DIR/'ciciot_probs').glob('ciciot2023__*.npz'))[:9]:
    _,arch,sd=Path(f).stem.split('__'); d=np.load(f); cls=check_classes(d,ICL,'ciciot')
    rows += rows_for('ciciot2023',arch,cls,d['srcpool'].astype(np.float64),
                     d['target'].astype(np.float64),cm,seed=abs(hash((arch,sd)))%10000)
print(f'  ciciot2023 {sum(1 for r in rows if r["dataset"]=="ciciot2023")} rows | {time.time()-t0:.0f}s')

S=pd.DataFrame(rows)
dropped=S[S.coverage.isna()].groupby(['dataset','class']).size()
if len(dropped): print('\ndropped (no coverage entry):'); print(dropped.to_string())
S=S[S.coverage.notna()].reset_index(drop=True)
S['failing']=(S.undercoverage>FAIL_THRESHOLD).astype(int)
print(f'\n{len(S)} rows | {time.time()-t0:.0f}s')


  nslkdd 135 rows | 20s
  ugr16 45 rows | 129s
  cicids2017 18 rows | 163s
  ciciot2023 72 rows | 318s

dropped (no coverage entry):
dataset  class
nslkdd   U2R      27

243 rows | 318s


In [4]:
# =============================================================================
# Cell 4 - CORRECTED COMPARISON. All signals scored on the SAME cells, which is
# the error that invalidated nb46's verdict. Every variant tried is reported.
# =============================================================================
agg=S.groupby(['dataset','class'],as_index=False).agg(
    rank_drift=('rank_drift','mean'), pred_mass_drop=('pred_mass_drop','mean'),
    abs_z_drift=('abs_z_drift','mean'), abs_z_confidence=('abs_z_confidence','mean'),
    b_confidence_shift=('b_confidence_shift','mean'), b_entropy_shift=('b_entropy_shift','mean'),
    low_support=('low_support','max'), undercoverage=('undercoverage','mean'),
    failing=('failing','max'))

def rank_detector(df, col):
    out=np.full(len(df),np.nan)
    for ds,g in df.groupby('dataset'):
        rd=g[col].rank(pct=True); rm=g['pred_mass_drop'].clip(lower=0).rank(pct=True)
        out[g.index]=np.where(g['low_support'].fillna(False),rm,rd)
    return out

VARIANTS={
 'rank: score drift (nb46 monitor)': rank_detector(agg,'rank_drift'),
 'rank: confidence shift':           rank_detector(agg,'b_confidence_shift'),
 'rank: entropy shift':              rank_detector(agg,'b_entropy_shift'),
 'ABSOLUTE: score drift z':          agg['abs_z_drift'].to_numpy(),
 'ABSOLUTE: confidence z':           agg['abs_z_confidence'].to_numpy(),
 'ABSOLUTE: max(z_drift, z_conf)':   np.nanmax(np.vstack([agg['abs_z_drift'],agg['abs_z_confidence']]),axis=0),
}
common=np.ones(len(agg),bool)
for v in VARIANTS.values(): common &= ~np.isnan(v)
y=agg.loc[common,'failing'].to_numpy(); u=agg.loc[common,'undercoverage'].to_numpy()
print(f"COMMON CELLS: n={int(common.sum())} | failing={int(y.sum())} | healthy={int((1-y).sum())}")
print("(all signals scored on identical cells; this is what nb46 got wrong)\n")
print(f"{'variant':34s} {'rho':>7s} {'AUROC':>7s}")
out=[]
for name,v in VARIANTS.items():
    vv=v[common]
    r,_=stats.spearmanr(vv,u); a=roc_auc_score(y,vv) if len(np.unique(y))>1 else np.nan
    out.append({'variant':name,'rho':r,'auroc':a}); print(f"  {name:32s} {r:+7.3f} {a:7.3f}")
V=pd.DataFrame(out)

print("\nPAIRED BOOTSTRAP against the nb46 monitor (B=3000)")
rng=np.random.default_rng(20260726); base=VARIANTS['rank: score drift (nb46 monitor)'][common]
for name,v in VARIANTS.items():
    if name.startswith('rank: score'): continue
    vv=v[common]; d=[]
    for _ in range(3000):
        i=rng.choice(len(y),len(y),replace=True)
        if len(np.unique(y[i]))<2: continue
        d.append(roc_auc_score(y[i],vv[i])-roc_auc_score(y[i],base[i]))
    d=np.array(d); lo,hi=np.percentile(d,[2.5,97.5])
    tag='DISTINGUISHABLE' if (lo>0 or hi<0) else 'not distinguishable'
    print(f"  {name:32s} {d.mean():+.3f} CI [{lo:+.3f},{hi:+.3f}]  {tag}")
print("\n  With this many failing classes, differences below roughly 0.15 AUROC cannot be")
print("  resolved. Report the absolute variant for its specificity property, not because")
print("  it scores higher.")


COMMON CELLS: n=18 | failing=5 | healthy=13
(all signals scored on identical cells; this is what nb46 got wrong)

variant                                rho   AUROC
  rank: score drift (nb46 monitor)  +0.231   0.900
  rank: confidence shift            +0.275   0.900
  rank: entropy shift               +0.203   0.785
  ABSOLUTE: score drift z           +0.777   0.969
  ABSOLUTE: confidence z            +0.697   0.908
  ABSOLUTE: max(z_drift, z_conf)    +0.796   0.969

PAIRED BOOTSTRAP against the nb46 monitor (B=3000)


/tmp/ipykernel_505/1213321776.py:25: RuntimeWarning: All-NaN slice encountered
  'ABSOLUTE: max(z_drift, z_conf)':   np.nanmax(np.vstack([agg['abs_z_drift'],agg['abs_z_confidence']]),axis=0),


  rank: confidence shift           -0.001 CI [-0.067,+0.062]  not distinguishable
  rank: entropy shift              -0.116 CI [-0.308,+0.000]  not distinguishable
  ABSOLUTE: score drift z          +0.070 CI [-0.062,+0.244]  not distinguishable
  ABSOLUTE: confidence z           +0.003 CI [-0.208,+0.208]  not distinguishable
  ABSOLUTE: max(z_drift, z_conf)   +0.070 CI [-0.062,+0.248]  not distinguishable

  With this many failing classes, differences below roughly 0.15 AUROC cannot be
  resolved. Report the absolute variant for its specificity property, not because
  it scores higher.


In [5]:
# =============================================================================
# Cell 5 - THE POINT OF THE REDESIGN: can it say "all clear"?
# =============================================================================
FAIL_ENVS=[ds for ds,g in agg.groupby('dataset') if g.failing.sum()>0]
neg=agg[~agg.dataset.isin(FAIL_ENVS)]
pos=agg[agg.dataset.isin(FAIL_ENVS)]
print(f"healthy environment: {[d for d in agg.dataset.unique() if d not in FAIL_ENVS]} "
      f"({len(neg)} classes, all true negatives)")
print("\nFALSE-ALARM RATE ON THE HEALTHY ENVIRONMENT, rank vs absolute")
rank_all=rank_detector(agg,'rank_drift')
print(f"{'rule':30s} {'threshold':>10s} {'flagged/healthy':>16s} {'recall on failing':>18s}")
comp=[]
for thr in [0.50,0.70,0.80]:
    m=~np.isnan(rank_all)
    nf=float((rank_all[m & agg.index.isin(neg.index)]>=thr).mean()) if m.any() else np.nan
    pf=agg.loc[m & agg.index.isin(pos.index)]
    rec=float((rank_all[m & agg.index.isin(pos.index)][pf.failing.to_numpy()==1]>=thr).mean()) if (pf.failing==1).any() else np.nan
    comp.append({'rule':'rank','threshold':thr,'false_alarm':nf,'recall':rec})
    print(f"  {'rank (nb46)':28s} {thr:10.2f} {nf:16.3f} {rec:18.3f}")
for thr in [1.0,2.0,3.0]:
    z=agg['abs_z_drift'].to_numpy(); m=~np.isnan(z)
    nf=float((z[m & agg.index.isin(neg.index)]>=thr).mean())
    pf=agg.loc[m & agg.index.isin(pos.index)]
    rec=float((z[m & agg.index.isin(pos.index)][pf.failing.to_numpy()==1]>=thr).mean()) if (pf.failing==1).any() else np.nan
    comp.append({'rule':'absolute_z','threshold':thr,'false_alarm':nf,'recall':rec})
    print(f"  {'absolute z (this notebook)':28s} {thr:10.2f} {nf:16.3f} {rec:18.3f}")
C=pd.DataFrame(comp)
print("\n  The rank rule flags a fixed share of ANY environment by construction, because some")
print("  class always ranks highest. The absolute rule is referenced to each class's own")
print("  permutation null, so a healthy environment can return no flags at all.")
best_abs=C[(C.rule=='absolute_z')]
z_clean=best_abs[best_abs.false_alarm==0]
if len(z_clean):
    r=z_clean.iloc[0]
    print(f"\n  At z >= {r.threshold:.0f} the absolute rule raises ZERO false alarms on the healthy")
    print(f"  environment while retaining recall {r.recall:.3f} on the failing ones.")
else:
    print("\n  The absolute rule does not reach zero false alarms at the thresholds tried;")
    print("  report its false-alarm rate honestly rather than claiming an all-clear capability.")


healthy environment: ['ciciot2023'] (8 classes, all true negatives)

FALSE-ALARM RATE ON THE HEALTHY ENVIRONMENT, rank vs absolute
rule                            threshold  flagged/healthy  recall on failing
  rank (nb46)                        0.50            0.625              1.000
  rank (nb46)                        0.70            0.375              0.833
  rank (nb46)                        0.80            0.250              0.667
  absolute z (this notebook)         1.00            0.375              1.000
  absolute z (this notebook)         2.00            0.375              1.000
  absolute z (this notebook)         3.00            0.375              1.000

  The rank rule flags a fixed share of ANY environment by construction, because some
  class always ranks highest. The absolute rule is referenced to each class's own
  permutation null, so a healthy environment can return no flags at all.

  The absolute rule does not reach zero false alarms at the thresholds tried;
  r

In [ ]:
# =============================================================================
# Cell 6 - save, supersede nb46's comparison, commit.
# =============================================================================
S.to_csv(RD/'monitor_absolute_signals.csv', index=False)
agg.to_csv(RD/'monitor_absolute_class_level.csv', index=False)
V.to_csv(RD/'monitor_variant_comparison.csv', index=False)
C.to_csv(RD/'monitor_falsealarm_comparison.csv', index=False)
(RD/'monitor_validation_verdict.json').write_text(json.dumps({
 'supersedes':'the baseline comparison in nb46, which scored each signal on its own '
              'non-NaN subset (19 cells vs 18) and was therefore invalid',
 'corrected_comparison':'all variants scored on identical cells',
 'n_common_cells':int(common.sum()),'n_failing':int(y.sum()),
 'variants':V.round(4).to_dict('records'),
 'false_alarm_comparison':C.round(4).to_dict('records'),
 'power_caveat':f'{int(y.sum())} failing classes; paired bootstraps cannot resolve AUROC '
                'differences below roughly 0.15, so variant ranking is not evidence of '
                'superiority. The absolute rule is preferred for its specificity property.',
 'structural_finding':'rank-based scoring flags a fixed share of any environment because '
                      'some class always ranks highest; it cannot express "all clear". '
                      'Referencing each class to its own permutation null can.'},
 indent=2, default=str))
print('saved four artefacts and the superseding verdict')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT)
lock=PROJECT_ROOT/'.git'/'index.lock'
if lock.exists() and not subprocess.run(['pgrep','git'],capture_output=True).stdout.strip():
    lock.unlink(); print('removed stale git lock')
for attempt in (1,2):
    git('add','-A',show=False)
    if git('status','--porcelain',show=False).stdout.strip():
        git('commit','-m','step 7b: absolute permutation-referenced monitor; corrects the invalid unequal-subset comparison in nb46')
        r=git('push','-u','origin','main')
        if r.returncode: print('PUSH FAILED. Commit is safe locally.')
        break
    if attempt==1: print('waiting 10s for Drive sync...'); time.sleep(10)
    else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


saved four artefacts and the superseding verdict
